In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import math

class HybridFUICF:
    def __init__(self, k_ucf=30, k_icf=10, gamma=0.5):
        self.k_ucf = k_ucf
        self.k_icf = k_icf
        self.gamma = gamma
        self.min_rating = 1
        self.max_rating = 5

        self.user_sim_matrix = None
        self.item_sim_matrix = None
        self.ratings_matrix = None
        self.user_means = None
        self.item_means = None
        self.global_mean = None

        self.user_id_map = {}
        self.item_id_map = {}
        self.id_user_map = {}
        self.id_item_map = {}

    def fit(self, train_file_path):
        print(f"Loading data from {train_file_path}...")
        df = pd.read_csv(train_file_path, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

        self.ratings_matrix = df.pivot(index='user_id', columns='item_id', values='rating')
        self.global_mean = df['rating'].mean()

        self.user_ids = self.ratings_matrix.index.tolist()
        self.item_ids = self.ratings_matrix.columns.tolist()

        self.user_id_map = {uid: i for i, uid in enumerate(self.user_ids)}
        self.item_id_map = {iid: i for i, iid in enumerate(self.item_ids)}
        self.id_user_map = {i: uid for i, uid in enumerate(self.user_ids)}
        self.id_item_map = {i: iid for i, iid in enumerate(self.item_ids)}

        print("Calculating Means...")
        self.user_means = self.ratings_matrix.mean(axis=1).fillna(0).values
        self.item_means = self.ratings_matrix.mean(axis=0).fillna(0).values

        print("Computing User-User Similarity...")
        matrix_user_centered = self.ratings_matrix.sub(self.ratings_matrix.mean(axis=1), axis=0).fillna(0)
        self.user_sim_matrix = cosine_similarity(matrix_user_centered)
        np.fill_diagonal(self.user_sim_matrix, 0)

        print("Computing Item-Item Similarity...")
        matrix_item_centered = self.ratings_matrix.sub(self.ratings_matrix.mean(axis=0), axis=1).fillna(0).T
        self.item_sim_matrix = cosine_similarity(matrix_item_centered)
        np.fill_diagonal(self.item_sim_matrix, 0)

        self.R = self.ratings_matrix.fillna(0).values
        print("Training Complete.")

    def predict_ucf(self, u_idx, i_idx):
        sim_scores = self.user_sim_matrix[u_idx]
        rated_users_indices = self.R[:, i_idx].nonzero()[0]

        if len(rated_users_indices) == 0:
            return self.user_means[u_idx]

        candidate_sims = sim_scores[rated_users_indices]
        candidate_indices = rated_users_indices

        if len(candidate_indices) > self.k_ucf:
            top_k_args = np.argsort(candidate_sims)[-self.k_ucf:]
            neighbors_indices = candidate_indices[top_k_args]
        else:
            neighbors_indices = candidate_indices

        numerator = 0
        denominator = 0

        for v_idx in neighbors_indices:
            sim_uv = self.user_sim_matrix[u_idx, v_idx]
            r_vi = self.R[v_idx, i_idx]
            mu_v = self.user_means[v_idx]
            numerator += sim_uv * (r_vi - mu_v)
            denominator += abs(sim_uv)

        if denominator == 0:
            return self.user_means[u_idx]

        return self.user_means[u_idx] + (numerator / denominator)

    def predict_icf(self, u_idx, i_idx):
        sim_scores = self.item_sim_matrix[i_idx]
        rated_items_indices = self.R[u_idx, :].nonzero()[0]

        if len(rated_items_indices) == 0:
            return self.item_means[i_idx]

        candidate_sims = sim_scores[rated_items_indices]
        candidate_indices = rated_items_indices

        if len(candidate_indices) > self.k_icf:
            top_k_args = np.argsort(candidate_sims)[-self.k_icf:]
            neighbors_indices = candidate_indices[top_k_args]
        else:
            neighbors_indices = candidate_indices

        numerator = 0
        denominator = 0

        for j_idx in neighbors_indices:
            sim_ij = self.item_sim_matrix[i_idx, j_idx]
            r_uj = self.R[u_idx, j_idx]
            mu_j = self.item_means[j_idx]
            numerator += sim_ij * (r_uj - mu_j)
            denominator += abs(sim_ij)

        if denominator == 0:
            return self.item_means[i_idx]

        return self.item_means[i_idx] + (numerator / denominator)

    def predict_hybrid(self, user_id, item_id):
        if user_id not in self.user_id_map or item_id not in self.item_id_map:
            return self.global_mean

        u_idx = self.user_id_map[user_id]
        i_idx = self.item_id_map[item_id]

        pred_ucf = self.predict_ucf(u_idx, i_idx)
        pred_icf = self.predict_icf(u_idx, i_idx)

        final_pred = (self.gamma * pred_ucf) + ((1 - self.gamma) * pred_icf)
        return np.clip(final_pred, self.min_rating, self.max_rating)

def compute_nmae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    return mae / 4.0

if __name__ == "__main__":
    base_dir = r"E:\NewDownloads\cf1\ml-100k"

    fold_nmae = []
    fold_mae = []

    print("\n==============================")
    print("5-FOLD EVALUATION (ML-100K)")
    print("==============================\n")

    for fold in range(1, 6):
        print(f"\n--- Fold {fold} ---")

        train_path = f"{base_dir}\\u{fold}.base"
        test_path = f"{base_dir}\\u{fold}.test"

        model = HybridFUICF(k_ucf=30, k_icf=10, gamma=0.5)
        model.fit(train_path)

        test_df = pd.read_csv(
            test_path,
            sep="\t",
            names=["user_id", "item_id", "rating", "timestamp"]
        )

        y_true, y_pred = [], []

        for _, row in test_df.iterrows():
            pred = model.predict_hybrid(row["user_id"], row["item_id"])
            y_true.append(row["rating"])
            y_pred.append(pred)

        mae = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
        nmae = compute_nmae(y_true, y_pred)

        fold_mae.append(mae)
        fold_nmae.append(nmae)

        print(f"MAE  (Fold {fold}): {mae:.4f}")
        print(f"NMAE (Fold {fold}): {nmae:.4f}")

    print("\n==============================")
    print("FINAL 5-FOLD RESULTS")
    print("==============================")
    for i, n in enumerate(fold_nmae, 1):
        print(f"Fold {i} NMAE: {n:.4f}")

    print("------------------------------")
    print(f"Average MAE :  {np.mean(fold_mae):.4f}")
    print(f"Average NMAE: {np.mean(fold_nmae):.4f}")



5-FOLD EVALUATION (ML-100K)


--- Fold 1 ---
Loading data from E:\NewDownloads\cf1\ml-100k\u1.base...
Calculating Means...
Computing User-User Similarity...
Computing Item-Item Similarity...
Training Complete.
MAE  (Fold 1): 0.7228
NMAE (Fold 1): 0.1807

--- Fold 2 ---
Loading data from E:\NewDownloads\cf1\ml-100k\u2.base...
Calculating Means...
Computing User-User Similarity...
Computing Item-Item Similarity...
Training Complete.
MAE  (Fold 2): 0.7112
NMAE (Fold 2): 0.1778

--- Fold 3 ---
Loading data from E:\NewDownloads\cf1\ml-100k\u3.base...
Calculating Means...
Computing User-User Similarity...
Computing Item-Item Similarity...
Training Complete.
MAE  (Fold 3): 0.7110
NMAE (Fold 3): 0.1778

--- Fold 4 ---
Loading data from E:\NewDownloads\cf1\ml-100k\u4.base...
Calculating Means...
Computing User-User Similarity...
Computing Item-Item Similarity...
Training Complete.
MAE  (Fold 4): 0.7094
NMAE (Fold 4): 0.1774

--- Fold 5 ---
Loading data from E:\NewDownloads\cf1\ml-100k\u5.base.

In [2]:
import pandas as pd
import numpy as np
import os

class UserBasedCF:
    def __init__(self):
        self.train_data = None
        self.user_item_matrix = None
        self.user_similarity_df = None
        self.user_means = None

    def load_data(self, base_file_path):
        if not os.path.exists(base_file_path):
            raise FileNotFoundError(f"File not found: {base_file_path}")

        print(f"  Loading training data: {os.path.basename(base_file_path)}...")

        try:
            self.train_data = pd.read_csv(base_file_path, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
        except:
            self.train_data = pd.read_csv(base_file_path, delim_whitespace=True, names=['user_id', 'item_id', 'rating', 'timestamp'])

        self.user_item_matrix = self.train_data.pivot(index='user_id', columns='item_id', values='rating')
        self.user_means = self.user_item_matrix.mean(axis=1)

    def compute_similarity(self):
        print("  Computing Pearson Correlation matrix...")
        matrix_centered = self.user_item_matrix.sub(self.user_means, axis=0)
        self.user_similarity_df = matrix_centered.T.corr(method='pearson')

    def predict(self, user_id, item_id, k=50):
        if item_id not in self.user_item_matrix.columns:
            return self.user_means.get(user_id, 3.0)

        if user_id not in self.user_similarity_df.index:
            return 3.0 

        sim_scores = self.user_similarity_df[user_id]
        item_ratings = self.user_item_matrix[item_id]

        valid_neighbors = sim_scores[
            (item_ratings.notna()) & 
            (sim_scores > 0) & 
            (sim_scores.index != user_id)
        ]

        if valid_neighbors.empty:
            return self.user_means[user_id]

        top_neighbors = valid_neighbors.sort_values(ascending=False).head(k)

        numerator = 0
        denominator = 0

        for neighbor_id, similarity in top_neighbors.items():
            neighbor_rating = self.user_item_matrix.loc[neighbor_id, item_id]
            neighbor_mean = self.user_means[neighbor_id]
            numerator += similarity * (neighbor_rating - neighbor_mean)
            denominator += abs(similarity)

        if denominator == 0:
            return self.user_means[user_id]

        prediction = self.user_means[user_id] + (numerator / denominator)
        return min(5, max(1, prediction))

    def evaluate(self, test_file_path):
        if not os.path.exists(test_file_path):
            raise FileNotFoundError(f"File not found: {test_file_path}")

        print(f"  Evaluating on test set: {os.path.basename(test_file_path)}...")

        try:
            test_data = pd.read_csv(test_file_path, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
        except:
            test_data = pd.read_csv(test_file_path, delim_whitespace=True, names=['user_id', 'item_id', 'rating', 'timestamp'])

        errors = []
        test_records = test_data.to_records(index=False)

        for record in test_records:
            user = record['user_id']
            item = record['item_id']
            actual = record['rating']
            predicted = self.predict(user, item)
            errors.append(abs(predicted - actual))

        mae = sum(errors) / len(errors)
        nmae = mae / 4.0

        print(f"  > Fold Results -> MAE: {mae:.4f} | NMAE: {nmae:.4f}")
        return mae, nmae

if __name__ == "__main__":
    DATA_DIRECTORY = r"E:\NewDownloads\cf1\ml-100k"

    fold_maes = []
    fold_nmaes = []

    print("\n================================================")
    print("STARTING 5-FOLD CROSS-VALIDATION")
    print("================================================\n")

    for i in range(1, 6):
        base_file = os.path.join(DATA_DIRECTORY, f"u{i}.base")
        test_file = os.path.join(DATA_DIRECTORY, f"u{i}.test")

        print(f"PROCESSING FOLD {i}/5")
        print("-" * 20)

        recommender = UserBasedCF()

        try:
            recommender.load_data(base_file)
            recommender.compute_similarity()
            mae, nmae = recommender.evaluate(test_file)
            fold_maes.append(mae)
            fold_nmaes.append(nmae)
        except FileNotFoundError as e:
            print(f"ERROR: {e}")
            break
        except Exception as e:
            print(f"An unexpected error occurred in Fold {i}: {e}")
            break

        print("\n")

    if fold_maes:
        avg_mae = sum(fold_maes) / len(fold_maes)
        avg_nmae = sum(fold_nmaes) / len(fold_nmaes)

        print("================================================")
        print("FINAL 5-FOLD CROSS-VALIDATION RESULTS")
        print("================================================")
        for i in range(len(fold_maes)):
            print(f"Fold {i+1}: MAE = {fold_maes[i]:.4f}, NMAE = {fold_nmaes[i]:.4f}")

        print("-" * 48)
        print(f"AVERAGE MAE  : {avg_mae:.4f}")
        print(f"AVERAGE NMAE : {avg_nmae:.4f}")
        print("================================================")



STARTING 5-FOLD CROSS-VALIDATION

PROCESSING FOLD 1/5
--------------------
  Loading training data: u1.base...
  Computing Pearson Correlation matrix...
  Evaluating on test set: u1.test...
  > Fold Results -> MAE: 0.7525 | NMAE: 0.1881


PROCESSING FOLD 2/5
--------------------
  Loading training data: u2.base...
  Computing Pearson Correlation matrix...
  Evaluating on test set: u2.test...
  > Fold Results -> MAE: 0.7426 | NMAE: 0.1856


PROCESSING FOLD 3/5
--------------------
  Loading training data: u3.base...
  Computing Pearson Correlation matrix...
  Evaluating on test set: u3.test...
  > Fold Results -> MAE: 0.7406 | NMAE: 0.1852


PROCESSING FOLD 4/5
--------------------
  Loading training data: u4.base...
  Computing Pearson Correlation matrix...
  Evaluating on test set: u4.test...
  > Fold Results -> MAE: 0.7390 | NMAE: 0.1848


PROCESSING FOLD 5/5
--------------------
  Loading training data: u5.base...
  Computing Pearson Correlation matrix...
  Evaluating on test set: 

In [3]:
import pandas as pd
import numpy as np
import os

class ItemBasedCF:
    def __init__(self, k_neighbors=20):
        self.k_neighbors = k_neighbors
        self.similarity_matrix = None
        self.ratings_matrix = None
        self.user_means = None
        self.item_means = None

    def fit(self, train_df):
        self.ratings_matrix = train_df.pivot(index='user_id', columns='item_id', values='rating')
        self.user_means = self.ratings_matrix.mean(axis=1)
        self.item_means = self.ratings_matrix.mean(axis=0)

        ratings_centered = self.ratings_matrix.sub(self.user_means, axis=0).fillna(0)
        item_matrix = ratings_centered.T

        dot_product = item_matrix.dot(item_matrix.T)
        norm = np.sqrt((item_matrix ** 2).sum(axis=1))
        similarity = dot_product / (np.outer(norm, norm) + 1e-9)

        self.similarity_matrix = pd.DataFrame(
            similarity,
            index=self.ratings_matrix.columns,
            columns=self.ratings_matrix.columns
        )

        np.fill_diagonal(self.similarity_matrix.values, 0)

    def predict(self, user_id, item_id):
        if item_id not in self.similarity_matrix.columns:
            return self.user_means.get(user_id, 3.0)

        if user_id not in self.ratings_matrix.index:
            return self.item_means.get(item_id, 3.0)

        user_ratings = self.ratings_matrix.loc[user_id].dropna()
        relevant_items = user_ratings.index.intersection(self.similarity_matrix.index)

        if len(relevant_items) == 0:
            return self.user_means[user_id]

        similarities = self.similarity_matrix.loc[item_id, relevant_items]

        if similarities.empty:
            return self.user_means[user_id]

        top_k_neighbors = similarities.nlargest(self.k_neighbors)
        relevant_ratings = user_ratings.loc[top_k_neighbors.index]

        numerator = (top_k_neighbors * relevant_ratings).sum()
        denominator = top_k_neighbors.abs().sum()

        if denominator == 0:
            return self.user_means[user_id]

        prediction = numerator / denominator
        return min(max(prediction, 1.0), 5.0)

def calculate_nmae(predictions, actuals, r_min=1, r_max=5):
    mae = np.mean(np.abs(predictions - actuals))
    nmae = mae / (r_max - r_min)
    return nmae, mae

def run_cross_validation(data_path):
    n_folds = 5
    nmae_scores = []

    print(f"{'Fold':<10} | {'MAE':<10} | {'NMAE':<10}")
    print("-" * 36)

    for i in range(1, n_folds + 1):
        train_file = os.path.join(data_path, f'u{i}.base')
        test_file = os.path.join(data_path, f'u{i}.test')

        cols = ['user_id', 'item_id', 'rating', 'timestamp']
        train_df = pd.read_csv(train_file, sep='\t', names=cols, encoding='latin-1')
        test_df = pd.read_csv(test_file, sep='\t', names=cols, encoding='latin-1')

        recommender = ItemBasedCF(k_neighbors=30)
        recommender.fit(train_df)

        preds = []
        actuals = []

        for _, row in test_df.iterrows():
            predicted_rating = recommender.predict(row['user_id'], row['item_id'])
            preds.append(predicted_rating)
            actuals.append(row['rating'])

        preds = np.array(preds)
        actuals = np.array(actuals)

        nmae, mae = calculate_nmae(preds, actuals, r_min=1, r_max=5)
        nmae_scores.append(nmae)

        print(f"Fold {i:<5} | {mae:.4f}     | {nmae:.4f}")

    avg_nmae = np.mean(nmae_scores)
    print("-" * 36)
    print(f"Average NMAE over {n_folds} folds: {avg_nmae:.4f}")

if __name__ == "__main__":
    YOUR_DATA_PATH = r"E:\NewDownloads\cf1\ml-100k"

    if os.path.exists(os.path.join(YOUR_DATA_PATH, 'u1.base')):
        run_cross_validation(YOUR_DATA_PATH)
    else:
        print(f"Error: Could not find 'u1.base' in path: {YOUR_DATA_PATH}")
        print("Please set YOUR_DATA_PATH to the folder containing the u1-u5 files.")


Fold       | MAE        | NMAE      
------------------------------------
Fold 1     | 0.9275     | 0.2319
Fold 2     | 0.8715     | 0.2179
Fold 3     | 0.8772     | 0.2193
Fold 4     | 0.8823     | 0.2206
Fold 5     | 0.9265     | 0.2316
------------------------------------
Average NMAE over 5 folds: 0.2242
